# 04. Huấn luyện và đánh giá Naive Bayes

Notebook này sử dụng đúng dữ liệu WELFake và các tập train/validation/test đã được tạo ở Notebook 05, giống CNN, RNN-LSTM và BERT.

**Quy ước nhãn WELFake:** `0 = Tin giả`, `1 = Tin thật`. Các chỉ số Precision/Recall/F1 dùng lớp dương `1` (Tin thật), thống nhất với các notebook 06–08.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib matplotlib seaborn

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount('/content/drive')

SEED = 42
DATA_DIR = "/content/drive/MyDrive/fake-news-detection/data/processed"
RESULT_DIR = "/content/drive/MyDrive/fake-news-detection/results"
os.makedirs(RESULT_DIR, exist_ok=True)

print("Đã import thư viện và kết nối Google Drive!")

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

for split_name, split_df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    if not {"text", "label"}.issubset(split_df.columns):
        raise ValueError(f"{split_name} phải có đủ hai cột text và label.")
    split_df["text"] = split_df["text"].fillna("").astype(str)
    split_df["label"] = pd.to_numeric(split_df["label"], errors="raise").astype(int)
    if not set(split_df["label"].unique()).issubset({0, 1}):
        raise ValueError(f"{split_name} chứa nhãn ngoài 0 và 1.")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Quy ước nhãn: 0 = Tin giả | 1 = Tin thật")
print("Phân bố nhãn Train:\n", train_df["label"].value_counts().sort_index())
print("Phân bố nhãn Test:\n", test_df["label"].value_counts().sort_index())

In [ ]:
X_train = train_df["text"]
y_train = train_df["label"]
X_test = test_df["text"]
y_test = test_df["label"]

# Chỉ fit TF-IDF trên tập Train để tránh rò rỉ dữ liệu.
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train:", X_train_tfidf.shape)
print("X_test:", X_test_tfidf.shape)

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

print("Đã huấn luyện và dự đoán Naive Bayes trên cùng tập dữ liệu WELFake.")

In [ ]:
results = {
    "Model": "Naive Bayes",
    "Accuracy": float(accuracy_score(y_test, y_pred)),
    "Precision": float(precision_score(y_test, y_pred, pos_label=1, zero_division=0)),
    "Recall": float(recall_score(y_test, y_pred, pos_label=1, zero_division=0)),
    "F1-score": float(f1_score(y_test, y_pred, pos_label=1, zero_division=0))
}

print("KẾT QUẢ ĐÁNH GIÁ NAIVE BAYES")
print(pd.DataFrame([results]).round(4))
print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Tin giả (0)", "Tin thật (1)"],
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Tin giả (0)", "Tin thật (1)"],
            yticklabels=["Tin giả (0)", "Tin thật (1)"])
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.title("Confusion Matrix - Naive Bayes")
plt.tight_layout()
plt.show()

In [ ]:
METRICS_PATH = os.path.join(RESULT_DIR, "naive_bayes_metrics.json")
MODEL_PATH = os.path.join(RESULT_DIR, "naive_bayes_model.joblib")
TFIDF_PATH = os.path.join(RESULT_DIR, "naive_bayes_tfidf.joblib")

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)
joblib.dump(model, MODEL_PATH)
joblib.dump(tfidf, TFIDF_PATH)

print("Đã lưu kết quả:", METRICS_PATH)
print("Đã lưu mô hình và TF-IDF vào:", RESULT_DIR)

In [ ]:
def predict_fake_news(news):
    clean_news = str(news).strip()
    news_vector = tfidf.transform([clean_news])
    prediction = int(model.predict(news_vector)[0])
    if prediction == 0:
        return "Dự đoán: TIN GIẢ (nhãn 0)"
    return "Dự đoán: TIN THẬT (nhãn 1)"

sample_news = "The government announced a new policy to improve education."
print(predict_fake_news(sample_news))